<a href="https://colab.research.google.com/github/harshag16/Harsh_Assessment_Link/blob/main/Assessment_Link_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
DATASET_PATH ="sample_shakespeare.txt"

SEQUENCE_LENGTH = 20
EMBEDDING_DIM = 128
LSTM_UNITS = 256

BATCH_SIZE = 128
EPOCHS = 30

VALIDATION_SPLIT = 0.1

MODEL_PATH = "best_lstm_model.keras"

# Random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [13]:
print("Loading dataset...")

with open("/content/sample_shakespeare.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Original text length:", len(text))

Loading dataset...
Original text length: 4027


In [14]:
def clean_text(text):


    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and keep alphabetic characters/numbers
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()


cleaned_text = clean_text(text)

print("Cleaned text length:", len(cleaned_text))
print("\nFirst 500 characters:")
print(cleaned_text[:500])


Cleaned text length: 3879

First 500 characters:
sonnet 18 shall i compare thee to a summer s day thou art more lovely and more temperate rough winds do shake the darling buds of may and summer s lease hath all too short a date sometime too hot the eye of heaven shines and often is his gold complexion dimm d and every fair from fair sometime declines by chance or nature s changing course untrimm d but thy eternal summer shall not fade nor lose possession of that fair thou ow st nor shall death brag thou wander st in his shade when in eternal l


In [15]:
tokenizer = Tokenizer(
    filters="",
    lower=False,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([cleaned_text])

total_words = len(tokenizer.word_index) + 1

print("\nVocabulary size:", total_words)

# Convert complete text to integer tokens
tokenized_text = tokenizer.texts_to_sequences([cleaned_text])[0]

print("Number of tokens:", len(tokenized_text))


Vocabulary size: 372
Number of tokens: 788


In [16]:
X = []
y = []

for i in range(SEQUENCE_LENGTH, len(tokenized_text)):
    sequence = tokenized_text[i - SEQUENCE_LENGTH:i]
    target = tokenized_text[i]

    X.append(sequence)
    y.append(target)

X = np.array(X)
y = np.array(y)

print("\nInput shape:", X.shape)
print("Target shape:", y.shape)


Input shape: (768, 20)
Target shape: (768,)


In [17]:
split_index = int(len(X) * (1 - VALIDATION_SPLIT))

X_train = X[:split_index]
y_train = y[:split_index]

X_val = X[split_index:]
y_val = y[split_index:]

print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))


Training samples: 691
Validation samples: 77


In [18]:
model = Sequential([
    # Convert word IDs into dense vectors
    Embedding(
        input_dim=total_words,
        output_dim=EMBEDDING_DIM,
        input_length=SEQUENCE_LENGTH
    ),

    # First LSTM layer
    LSTM(
        LSTM_UNITS,
        return_sequences=True
    ),

    Dropout(0.2),

    # Second LSTM layer
    LSTM(LSTM_UNITS),

    Dropout(0.2),

    # Predict probability for every word in vocabulary
    Dense(
        total_words,
        activation="softmax"
    )
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [19]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nModel architecture:")
model.summary()


Model architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    MODEL_PATH,
    monitor="val_loss",
    save_best_only=True
)

In [21]:
print("\nStarting training...")

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping,
        checkpoint
    ],
    verbose=1
)

print("\nTraining completed.")


Starting training...
Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 498ms/step - accuracy: 0.0275 - loss: 5.9138 - val_accuracy: 0.0779 - val_loss: 5.8848
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 653ms/step - accuracy: 0.0333 - loss: 5.7285 - val_accuracy: 0.0779 - val_loss: 5.7370
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 393ms/step - accuracy: 0.0333 - loss: 5.5157 - val_accuracy: 0.0390 - val_loss: 5.9107
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 397ms/step - accuracy: 0.0289 - loss: 5.4649 - val_accuracy: 0.0390 - val_loss: 6.0799
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 455ms/step - accuracy: 0.0333 - loss: 5.4368 - val_accuracy: 0.0390 - val_loss: 6.2515

Training completed.


In [23]:
def generate_text(seed_text, next_words=50, temperature=1.0):


    generated_text = seed_text.lower()

    for _ in range(next_words):

        # Convert seed text into token IDs
        token_list = tokenizer.texts_to_sequences(
            [generated_text]
        )[0]

        # Keep only the last SEQUENCE_LENGTH tokens
        token_list = token_list[-SEQUENCE_LENGTH:]

        # Pad sequence if it contains fewer tokens
        token_list = pad_sequences(
            [token_list],
            maxlen=SEQUENCE_LENGTH,
            padding="pre"
        )

        # Predict probabilities for the next word
        predictions = model.predict(
            token_list,
            verbose=0
        )[0]

        # Apply temperature
        predictions = np.asarray(predictions).astype("float64")

        predictions = np.log(
            predictions + 1e-8
        ) / temperature

        probabilities = np.exp(predictions)

        probabilities = probabilities / np.sum(
            probabilities
        )

        # Sample a word according to probabilities
        predicted_id = np.random.choice(
            len(probabilities),
            p=probabilities
        )

        # Convert token ID back to word
        output_word = None

        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                output_word = word
                break

        if output_word is None:
            continue

        generated_text += " " + output_word

    return generated_text

In [24]:
print("\n" + "=" * 70)
print("GENERATED TEXT")
print("=" * 70)


seeds = [
    "to be or not to be",
    "the king shall",
    "love is",
    "o my lord"
]

for seed in seeds:

    print("\nSeed:", seed)

    generated = generate_text(
        seed_text=seed,
        next_words=50,
        temperature=0.8
    )

    print("Generated:")
    print(generated)


GENERATED TEXT

Seed: to be or not to be
Generated:
to be or not to be it countrymen shines are and and to lov time than to after hymns and and and i caesar with i shall and i of it shines and all man to on and to rub bury cries i and every my and love to treads the red s this so and

Seed: the king shall
Generated:
the king shall you lark coil whether compare consummation and the to that it s finds that s her and look and creeps date and to possess what than 18 to is and height st is to s is than yet hairs from and such was her buds love caesar with to and

Seed: love is
Generated:
love is to let d man troubles is love beweep i and is and heir break are writ one and damask so curse well is and the my these unknown to this with the and is dream is caesar myself of oft lend the love i i to then his to i

Seed: o my lord
Generated:
o my lord say is and day last s wires look i buds with may compare sleep and which is and to on every to caesar the mistress and fortune that flesh and is an

In [26]:
print("\n" + "=" * 70)
print("FINAL MODEL EVALUATION")
print("=" * 70)

val_loss, val_accuracy = model.evaluate(
    X_val,
    y_val,
    verbose=0
)

print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


FINAL MODEL EVALUATION
Validation Loss: 5.736952781677246
Validation Accuracy: 0.07792207598686218


In [27]:
import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("\nTokenizer saved as tokenizer.pkl")
print("Model saved as", MODEL_PATH)


Tokenizer saved as tokenizer.pkl
Model saved as best_lstm_model.keras
